# ColorCLIP Model Inference Demo

Load the best 129-color and 15-color models, feed them common color names,
and see what the model retrieves. For each input color we show:
- The actual color name and its mean RGB
- The model's top-1 predicted name (highest cosine similarity)
- The cosine similarity score
- A visual swatch of both the true and predicted colors

In [1]:
import sys, os, re, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path

# Add project root so we can import ML modules
PROJECT_ROOT = Path(os.getcwd()).parent if 'analysis' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from ML.models.clip_models import ColorCLIPModel
from ML.embeddings import BagOfWordsEmbedder, rgb_to_oklch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')

Project root: c:\Users\LuizMarques\Downloads\Projects\Projects\colorsurvey
Device: cpu


## 1. Find Best Models from Experiment Results

In [2]:
results = pd.read_csv(PROJECT_ROOT / '5th_experiments' / 'experiment_results.csv')

# Best 129-color model (lowest rank = best R@1)
best_129 = results[results['Colors'] == 129].sort_values('Rank').iloc[0]
# Best 15-color model
best_15 = results[results['Colors'] == 15].sort_values('Rank').iloc[0]

print('=== Best 129-color model ===')
print(f"  Experiment: {best_129['Experiment']}")
print(f"  R@1={best_129['R_at_1']:.4f}  Delta_E={best_129['Delta_E']:.4f}  Color_Space={best_129['Color_Space']}")
print()
print('=== Best 15-color model ===')
print(f"  Experiment: {best_15['Experiment']}")
print(f"  R@1={best_15['R_at_1']:.4f}  Delta_E={best_15['Delta_E']:.4f}  Color_Space={best_15['Color_Space']}")

=== Best 129-color model ===
  Experiment: clip_rgb_129c_dim32_t0.03_lr0.003_wd0.0
  R@1=0.0074  Delta_E=0.3089  Color_Space=rgb

=== Best 15-color model ===
  Experiment: clip_oklch_15c_dim64_t0.07_lr0.001_wd0.1
  R@1=0.0030  Delta_E=0.2831  Color_Space=oklch


## 2. Load & Preprocess Data (same pipeline as training)

In [3]:
# Load CSV and apply the same treatment as analysis.ipynb
df_raw = pd.read_csv(PROJECT_ROOT / 'mainsurvey_data_with_space.csv')

# Normalize: lowercase, replace dashes/underscores with spaces, strip
df_raw['colorname'] = (
    df_raw['colorname']
    .str.replace(r'[\-\_]', ' ', regex=True)
    .str.lower()
    .str.strip()
)

# Keep only alphabetic+space names
df_raw = df_raw[df_raw['colorname'].str.match(r'^[a-zA-Z\s]+$')]

# Remove single chars, numbers, non-colors
df_raw = df_raw[~df_raw['colorname'].str.match(r"^(?:[0-9]|[a-zA-Z]|\\?|'')$")]

non_colors_path = PROJECT_ROOT / 'analysis' / 'non_colors.txt'
if non_colors_path.exists():
    with open(non_colors_path) as f:
        non_colors = [line.strip() for line in f if line.strip()]
    df_raw = df_raw[~df_raw['colorname'].str.lower().isin(non_colors)]

# Keep >= 10 entries
color_counts = df_raw['colorname'].value_counts()
df_raw = df_raw[df_raw['colorname'].isin(color_counts[color_counts >= 10].index)]

print(f'After filtering: {len(df_raw):,} rows, {df_raw["colorname"].nunique():,} unique color names')

After filtering: 3,125,903 rows, 7,116 unique color names


## 3. Helper: Reconstruct Training Pipeline & Load Model

In [4]:
def reconstruct_pipeline(df, top_n, color_space, vocab_size=100, seed=42):
    """
    Reproduce the exact same data pipeline used during training so
    the BoW vocabulary order matches the model weights.
    Returns: (label_encoder, bow_embedder, class_mean_rgb, class_names)
    """
    # Step 1: Filter to top N colors (same as load_and_preprocess_data)
    counts = df['colorname'].value_counts()
    top_colors = counts.nlargest(top_n).index
    df_f = df[df['colorname'].isin(top_colors)].copy()

    # Step 2: Prepare features and labels
    X = df_f[['r', 'g', 'b']].values.astype(np.float32) / 255.0
    le = LabelEncoder()
    y = le.fit_transform(df_f['colorname'].values)

    # Step 3: Train/test split with same seed
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    # Step 4: Fit BoW on training labels
    y_text_train = le.inverse_transform(y_train)
    bow = BagOfWordsEmbedder(top_n_words=vocab_size)
    bow.fit(y_text_train.tolist())

    # Step 5: Compute mean RGB per class (from full filtered data)
    class_mean_rgb = (
        df_f.groupby('colorname')[['r', 'g', 'b']]
        .mean()
        .round()
        .astype(int)
    )

    return le, bow, class_mean_rgb, le.classes_


def load_model(experiment_name, vocab_size, embed_dim):
    """Load a saved ColorCLIP checkpoint."""
    path = PROJECT_ROOT / '5th_experiments' / 'models' / experiment_name / 'color_clip_model.pth'
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model = ColorCLIPModel(vocab_size=vocab_size, embed_dim=embed_dim).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, ckpt.get('config', {})

print('Helpers ready.')

Helpers ready.


## 4. Inference Function

In [5]:
def run_inference(model, le, bow_embedder, class_mean_rgb, color_space, test_names=None):
    """
    For each color name:
      1. Look up its mean RGB
      2. Encode the color through the color tower
      3. Encode ALL class names through the text tower
      4. Compute cosine similarities and find the best match
    
    Returns a DataFrame with: true_name, true_rgb, predicted_name, predicted_rgb,
                              cosine_sim, rank
    """
    all_names = list(le.classes_)
    if test_names is None:
        test_names = all_names

    # Pre-encode ALL text labels (build the "text gallery")
    bow_all = bow_embedder.transform(all_names)
    bow_tensor = torch.FloatTensor(bow_all).to(DEVICE)
    with torch.no_grad():
        text_embeds = model.encode_text(bow_tensor)  # (num_classes, D)

    rows = []
    for name in test_names:
        if name not in class_mean_rgb.index:
            continue

        rgb = class_mean_rgb.loc[name][['r', 'g', 'b']].values.astype(float)

        # Convert to model input space
        rgb_norm = rgb / 255.0
        if color_space == 'oklch':
            color_input = rgb_to_oklch(rgb.reshape(1, 3), normalize=True)  # expects 0-255
        else:
            color_input = rgb_norm.reshape(1, 3)

        color_tensor = torch.FloatTensor(color_input).to(DEVICE)
        with torch.no_grad():
            color_embed = model.encode_color(color_tensor)  # (1, D)

        # Cosine similarity (embeddings are already L2-normalized)
        sims = (color_embed @ text_embeds.t()).squeeze(0).cpu().numpy()  # (num_classes,)

        # Rank of the correct label
        true_idx = list(le.classes_).index(name)
        true_sim = sims[true_idx]
        rank = (sims > true_sim).sum() + 1  # 1-based rank

        # Best match
        best_idx = sims.argmax()
        pred_name = all_names[best_idx]
        pred_sim = sims[best_idx]
        pred_rgb = class_mean_rgb.loc[pred_name][['r', 'g', 'b']].values.astype(int) if pred_name in class_mean_rgb.index else [0,0,0]

        rows.append({
            'True Name': name,
            'True RGB': f'({int(rgb[0])}, {int(rgb[1])}, {int(rgb[2])})',
            'Predicted Name': pred_name,
            'Pred RGB': f'({pred_rgb[0]}, {pred_rgb[1]}, {pred_rgb[2]})' if isinstance(pred_rgb, np.ndarray) else str(pred_rgb),
            'Cosine Sim (pred)': round(float(pred_sim), 4),
            'Cosine Sim (true)': round(float(true_sim), 4),
            'True Rank': int(rank),
            'Correct': name == pred_name,
            '_true_r': int(rgb[0]), '_true_g': int(rgb[1]), '_true_b': int(rgb[2]),
            '_pred_r': int(pred_rgb[0]) if isinstance(pred_rgb, np.ndarray) else 0,
            '_pred_g': int(pred_rgb[1]) if isinstance(pred_rgb, np.ndarray) else 0,
            '_pred_b': int(pred_rgb[2]) if isinstance(pred_rgb, np.ndarray) else 0,
        })

    return pd.DataFrame(rows)

print('Inference function ready.')

Inference function ready.


## 5. Display Helper with Color Swatches

In [6]:
def color_swatch(r, g, b, size=20):
    """HTML for a colored square."""
    return f'<div style="display:inline-block;width:{size}px;height:{size}px;background:rgb({r},{g},{b});border:1px solid #ccc;vertical-align:middle;margin-right:4px;"></div>'

def display_results(df_res, title):
    """Display a styled HTML table with color swatches."""
    from IPython.display import HTML, display

    n_correct = df_res['Correct'].sum()
    n_total = len(df_res)
    avg_rank = df_res['True Rank'].mean()
    
    html = f'<h3 style="text-align:left;">{title}</h3>'
    html += f'<p style="text-align:left;">Correct matches: <b>{n_correct}/{n_total}</b> ({100*n_correct/n_total:.1f}%) &nbsp; | &nbsp; Mean true rank: <b>{avg_rank:.1f}</b> / {n_total}</p>'
    html += '<table style="border-collapse:collapse;font-size:13px;margin-left:0;margin-right:auto;">'
    html += '<tr style="background:#000;color:#fff;">'
    html += '<th style="padding:6px;border:1px solid #555;text-align:left;">Input RGB</th>'
    html += '<th style="padding:6px;border:1px solid #555;text-align:left;">True Name</th>'
    html += '<th style="padding:6px;border:1px solid #555;text-align:left;">Output Color Name</th>'
    html += '<th style="padding:6px;border:1px solid #555;text-align:left;">Output RGB (REFERENCE)</th>'
    html += '<th style="padding:6px;border:1px solid #555;text-align:left;">Cos Sim (pred)</th>'
    html += '<th style="padding:6px;border:1px solid #555;text-align:left;">Cos Sim (true)</th>'
    html += '<th style="padding:6px;border:1px solid #555;text-align:left;">True Rank</th>'
    html += '</tr>'

    for _, row in df_res.iterrows():
        bg = '#e8ffe8' if row['Correct'] else '#fff'
        html += f'<tr style="background:{bg};">'
        html += f'<td style="padding:6px;border:1px solid #ddd;text-align:left;">{color_swatch(row["_true_r"], row["_true_g"], row["_true_b"])} {row["True RGB"]}</td>'
        html += f'<td style="padding:6px;border:1px solid #ddd;text-align:left;"><b>{row["True Name"]}</b></td>'
        html += f'<td style="padding:6px;border:1px solid #ddd;text-align:left;"><b>{row["Predicted Name"]}</b></td>'
        html += f'<td style="padding:6px;border:1px solid #ddd;text-align:left;">{color_swatch(row["_pred_r"], row["_pred_g"], row["_pred_b"])} {row["Pred RGB"]}</td>'
        html += f'<td style="padding:6px;border:1px solid #ddd;text-align:center;">{row["Cosine Sim (pred)"]:.4f}</td>'
        html += f'<td style="padding:6px;border:1px solid #ddd;text-align:center;">{row["Cosine Sim (true)"]:.4f}</td>'
        html += f'<td style="padding:6px;border:1px solid #ddd;text-align:center;">{row["True Rank"]}</td>'
        html += '</tr>'

    html += '</table>'
    display(HTML(html))

print('Display helper ready.')

Display helper ready.


## 6. Run: Best 129-Color Model

In [7]:
# Reconstruct pipeline for 129 colors
exp_129 = best_129['Experiment']
cs_129 = best_129['Color_Space']
dim_129 = int(best_129['Embed_Dim'])

le_129, bow_129, mean_rgb_129, names_129 = reconstruct_pipeline(
    df_raw, top_n=129, color_space=cs_129
)
model_129, cfg_129 = load_model(exp_129, bow_129.vocab_size, dim_129)

print(f'Loaded {exp_129}')
print(f'Vocab size: {bow_129.vocab_size}, Embed dim: {dim_129}, Classes: {len(names_129)}')

Built BoW vocabulary with 83 words
Loaded clip_rgb_129c_dim32_t0.03_lr0.003_wd0.0
Vocab size: 83, Embed dim: 32, Classes: 129


In [8]:
# Run inference on ALL 129 color classes
df_129 = run_inference(model_129, le_129, bow_129, mean_rgb_129, cs_129)
display_results(df_129.sort_values('True Name'), f'129-Color Model: {exp_129}')

Input RGB,True Name,Output Color Name,Output RGB (REFERENCE),Cos Sim (pred),Cos Sim (true),True Rank
"(62, 206, 189)",aqua,turquoise,"(54, 195, 178)",0.3528,0.3457,5
"(58, 199, 177)",aquamarine,teal,"(57, 179, 159)",0.3553,0.3367,6
"(97, 123, 49)",army green,army green,"(97, 123, 49)",0.3913,0.3913,1
"(121, 193, 228)",baby blue,baby blue,"(121, 193, 228)",0.3729,0.3729,1
"(211, 192, 133)",beige,tan,"(203, 172, 103)",0.4094,0.4045,2
"(36, 33, 35)",black,black,"(36, 33, 35)",0.4748,0.4748,1
"(55, 97, 198)",blue,blue,"(55, 97, 198)",0.4315,0.4315,1
"(108, 138, 159)",blue gray,blue grey,"(110, 140, 161)",0.4141,0.4121,3
"(51, 169, 140)",blue green,teal,"(57, 179, 159)",0.3686,0.3432,2
"(110, 140, 161)",blue grey,blue grey,"(110, 140, 161)",0.4133,0.4133,1


## 7. Run: Best 15-Color Model

In [9]:
# Reconstruct pipeline for 15 colors
exp_15 = best_15['Experiment']
cs_15 = best_15['Color_Space']
dim_15 = int(best_15['Embed_Dim'])

le_15, bow_15, mean_rgb_15, names_15 = reconstruct_pipeline(
    df_raw, top_n=15, color_space=cs_15
)
model_15, cfg_15 = load_model(exp_15, bow_15.vocab_size, dim_15)

print(f'Loaded {exp_15}')
print(f'Vocab size: {bow_15.vocab_size}, Embed dim: {dim_15}, Classes: {len(names_15)}')

Built BoW vocabulary with 14 words
Loaded clip_oklch_15c_dim64_t0.07_lr0.001_wd0.1
Vocab size: 14, Embed dim: 64, Classes: 15


In [10]:
# Run inference on ALL 15 color classes
df_15 = run_inference(model_15, le_15, bow_15, mean_rgb_15, cs_15)
display_results(df_15.sort_values('True Name'), f'15-Color Model: {exp_15}')

Input RGB,True Name,Output Color Name,Output RGB (REFERENCE),Cos Sim (pred),Cos Sim (true),True Rank
"(55, 97, 198)",blue,blue,"(55, 97, 198)",0.6880,0.6880,1
"(136, 90, 44)",brown,brown,"(136, 90, 44)",0.9368,0.9368,1
"(74, 185, 68)",green,green,"(74, 185, 68)",0.7684,0.7684,1
"(144, 148, 143)",grey,grey,"(144, 148, 143)",0.9255,0.9255,1
"(99, 184, 223)",light blue,sky blue,"(90, 181, 226)",0.4843,0.4463,3
"(118, 224, 107)",light green,green,"(74, 185, 68)",0.5862,0.3646,2
"(200, 39, 152)",magenta,magenta,"(200, 39, 152)",0.5031,0.5031,1
"(228, 122, 38)",orange,orange,"(228, 122, 38)",0.8783,0.8783,1
"(227, 91, 169)",pink,pink,"(227, 91, 169)",0.7272,0.7272,1
"(142, 53, 173)",purple,purple,"(142, 53, 173)",0.7087,0.7087,1


## 8. Summary Statistics

In [11]:
from IPython.display import HTML, display

def summarize(df_res, label):
    n = len(df_res)
    correct = df_res['Correct'].sum()
    acc = 100 * correct / n
    random_acc = 100 / n
    mean_rank = df_res['True Rank'].mean()
    median_rank = df_res['True Rank'].median()
    mean_cos = df_res['Cosine Sim (true)'].mean()
    mean_pred_cos = df_res['Cosine Sim (pred)'].mean()
    return {
        'Model': label,
        'Classes': n,
        'Correct': f'{correct}/{n}',
        'Accuracy': f'{acc:.1f}%',
        'Random Baseline': f'{random_acc:.2f}%',
        'Mean True Rank': f'{mean_rank:.1f}',
        'Median True Rank': f'{median_rank:.0f}',
        'Mean Cos (true)': f'{mean_cos:.4f}',
        'Mean Cos (pred)': f'{mean_pred_cos:.4f}',
    }

summary = pd.DataFrame([
    summarize(df_129, f'129-color ({exp_129})'),
    summarize(df_15, f'15-color ({exp_15})'),
])
display(HTML(summary.to_html(index=False)))

Model,Classes,Correct,Accuracy,Random Baseline,Mean True Rank,Median True Rank,Mean Cos (true),Mean Cos (pred)
129-color (clip_rgb_129c_dim32_t0.03_lr0.003_wd0.0),129,57/129,44.2%,0.78%,2.0,2,0.3698,0.3899
15-color (clip_oklch_15c_dim64_t0.07_lr0.001_wd0.1),15,12/15,80.0%,6.67%,1.3,1,0.6533,0.7001


## 9. Random Baseline: Untrained Model Comparison

Compare against randomly initialized (untrained) models with the same architecture to confirm the trained models genuinely learned something.

In [12]:
# Create randomly initialized models (same architecture, no training)
torch.manual_seed(999)  # reproducible random baseline

# Random 129-color model
random_model_129 = ColorCLIPModel(vocab_size=bow_129.vocab_size, embed_dim=dim_129).to(DEVICE)
random_model_129.eval()
df_rand_129 = run_inference(random_model_129, le_129, bow_129, mean_rgb_129, cs_129)

# Random 15-color model
random_model_15 = ColorCLIPModel(vocab_size=bow_15.vocab_size, embed_dim=dim_15).to(DEVICE)
random_model_15.eval()
df_rand_15 = run_inference(random_model_15, le_15, bow_15, mean_rgb_15, cs_15)

# Compare
comparison = pd.DataFrame([
    summarize(df_129, 'Trained 129-color'),
    summarize(df_rand_129, 'Random 129-color (untrained)'),
    summarize(df_15, 'Trained 15-color'),
    summarize(df_rand_15, 'Random 15-color (untrained)'),
])
display(HTML(comparison.to_html(index=False)))

print()
n_correct_rand_129 = df_rand_129['Correct'].sum()
n_correct_rand_15 = df_rand_15['Correct'].sum()
print(f'129-color: Trained gets {df_129["Correct"].sum()}/129 correct vs Random {n_correct_rand_129}/129')
print(f'15-color:  Trained gets {df_15["Correct"].sum()}/15 correct vs Random {n_correct_rand_15}/15')

Model,Classes,Correct,Accuracy,Random Baseline,Mean True Rank,Median True Rank,Mean Cos (true),Mean Cos (pred)
Trained 129-color,129,57/129,44.2%,0.78%,2.0,2,0.3698,0.3899
Random 129-color (untrained),129,1/129,0.8%,0.78%,64.2,62,-0.0180,0.4916
Trained 15-color,15,12/15,80.0%,6.67%,1.3,1,0.6533,0.7001
Random 15-color (untrained),15,1/15,6.7%,6.67%,7.8,8,-0.0153,0.1032



129-color: Trained gets 57/129 correct vs Random 1/129
15-color:  Trained gets 12/15 correct vs Random 1/15


## 10. Why R@1 Was Misleading

The training evaluation computed R@1 on **individual test samples** (N≈5000), where many samples share the same class label and thus identical text embeddings. Even with perfect class-level discrimination, the sample-level R@1 is bounded by ≈1/(samples_per_class).

The per-class centroid evaluation above reveals the model **does** learn meaningful color→name associations, but this signal is diluted in the sample-level metric.

In [13]:
# Demonstrate the R@1 dilution effect
# The training evaluation subsamples to max_eval_samples (default 5000) 
# and computes R@1 on the diagonal of the N×N similarity matrix.
# With K classes and N samples, avg samples/class = N/K.
# Even with PERFECT class discrimination, R@1 ≈ 1/(N/K) = K/N

max_eval_samples = 5000  # default in color_clip_trainer.py

avg_per_class_129 = max_eval_samples / 129
avg_per_class_15 = max_eval_samples / 15

print("Sample-level R@1 upper bound (with max_eval_samples=5000):")
print(f"  129-color: {max_eval_samples} samples, ~{avg_per_class_129:.0f} per class → R@1 ≤ {1/avg_per_class_129:.4f} ({100/avg_per_class_129:.2f}%)")
print(f"  15-color:  {max_eval_samples} samples, ~{avg_per_class_15:.0f} per class → R@1 ≤ {1/avg_per_class_15:.4f} ({100/avg_per_class_15:.2f}%)")
print()
print("Observed R@1 from training evaluation:")
print(f"  129-color: {best_129['R_at_1']:.4f} ({100*best_129['R_at_1']:.2f}%)")
print(f"  15-color:  {best_15['R_at_1']:.4f} ({100*best_15['R_at_1']:.2f}%)")
print()
print("Per-class centroid accuracy (this notebook):")
print(f"  129-color: {100*df_129['Correct'].mean():.1f}%")
print(f"  15-color:  {100*df_15['Correct'].mean():.1f}%")
print()
print("Interpretation:")
print(f"  129-color: R@1=0.74% vs upper bound 2.58% → model achieves ~{0.0074/0.0258*100:.0f}% of theoretical max")
print(f"  15-color:  R@1=0.30% vs upper bound 0.30% → model achieves ~100% of theoretical max!")
print()
print("→ The sample-level R@1 is misleadingly low due to same-class duplicates.")
print("  At the class centroid level, the models learn strong color→name associations.")

Sample-level R@1 upper bound (with max_eval_samples=5000):
  129-color: 5000 samples, ~39 per class → R@1 ≤ 0.0258 (2.58%)
  15-color:  5000 samples, ~333 per class → R@1 ≤ 0.0030 (0.30%)

Observed R@1 from training evaluation:
  129-color: 0.0074 (0.74%)
  15-color:  0.0030 (0.30%)

Per-class centroid accuracy (this notebook):
  129-color: 44.2%
  15-color:  80.0%

Interpretation:
  129-color: R@1=0.74% vs upper bound 2.58% → model achieves ~29% of theoretical max
  15-color:  R@1=0.30% vs upper bound 0.30% → model achieves ~100% of theoretical max!

→ The sample-level R@1 is misleadingly low due to same-class duplicates.
  At the class centroid level, the models learn strong color→name associations.


## 11. Evaluate All 432 Models (Centroid Accuracy)

Load every model checkpoint, run centroid-based inference, and collect per-model
accuracy, mean rank, and cosine similarity into a single CSV.

In [14]:
import time, re as _re
from pathlib import Path as _Path

models_dir = PROJECT_ROOT / '5th_experiments' / 'models'
model_folders = sorted([d.name for d in models_dir.iterdir() if d.is_dir()])
print(f'Found {len(model_folders)} model folders')

# Parse hyperparams from folder name: clip_{cs}_{n}c_dim{d}_t{t}_lr{lr}_wd{wd}
def parse_experiment_name(name):
    m = _re.match(
        r'clip_(\w+?)_(\d+)c_dim(\d+)_t([\d.]+)_lr([\d.]+)_wd([\d.]+)', name
    )
    if not m:
        return None
    return {
        'color_space': m.group(1),
        'num_colors': int(m.group(2)),
        'embed_dim': int(m.group(3)),
        'temperature': float(m.group(4)),
        'lr': float(m.group(5)),
        'weight_decay': float(m.group(6)),
    }

# Pre-build pipelines for the two palette sizes (only need to do this once)
pipelines = {}
for n_colors in [15, 129]:
    # We need one pipeline per (n_colors, color_space) combo
    for cs in ['oklch', 'rgb']:
        key = (n_colors, cs)
        le, bow, mean_rgb, names = reconstruct_pipeline(
            df_raw, top_n=n_colors, color_space=cs
        )
        pipelines[key] = (le, bow, mean_rgb, names)

print(f'Built {len(pipelines)} pipelines')
print('Keys:', list(pipelines.keys()))

Found 432 model folders
Built BoW vocabulary with 14 words
Built BoW vocabulary with 14 words
Built BoW vocabulary with 83 words
Built BoW vocabulary with 83 words
Built 4 pipelines
Keys: [(15, 'oklch'), (15, 'rgb'), (129, 'oklch'), (129, 'rgb')]


In [15]:
all_results = []
t0 = time.time()

for i, folder in enumerate(model_folders):
    hp = parse_experiment_name(folder)
    if hp is None:
        print(f'  Skipping {folder} (could not parse name)')
        continue

    key = (hp['num_colors'], hp['color_space'])
    le, bow, mean_rgb, names = pipelines[key]

    # Load model
    try:
        model, _ = load_model(folder, bow.vocab_size, hp['embed_dim'])
    except Exception as e:
        print(f'  ERROR loading {folder}: {e}')
        all_results.append({**hp, 'experiment': folder, 'error': str(e)})
        continue

    # Run centroid inference
    df_res = run_inference(model, le, bow, mean_rgb, hp['color_space'])

    n = len(df_res)
    correct = int(df_res['Correct'].sum())

    all_results.append({
        'experiment': folder,
        **hp,
        'centroid_accuracy': correct / n if n > 0 else 0,
        'correct': correct,
        'total_classes': n,
        'mean_true_rank': df_res['True Rank'].mean(),
        'median_true_rank': df_res['True Rank'].median(),
        'mean_cos_true': df_res['Cosine Sim (true)'].mean(),
        'mean_cos_pred': df_res['Cosine Sim (pred)'].mean(),
    })

    # Progress every 50 models
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f'  [{i+1}/{len(model_folders)}] {elapsed:.0f}s elapsed')

elapsed = time.time() - t0
print(f'\nDone: {len(all_results)} models evaluated in {elapsed:.1f}s')

df_all = pd.DataFrame(all_results)
df_all.head()

  [50/432] 18s elapsed
  [100/432] 37s elapsed
  [150/432] 40s elapsed
  [200/432] 42s elapsed
  [250/432] 51s elapsed
  [300/432] 66s elapsed
  [350/432] 75s elapsed
  [400/432] 77s elapsed

Done: 432 models evaluated in 78.6s


,experiment,color_space,num_colors,embed_dim,temperature,lr,weight_decay,centroid_accuracy,correct,total_classes,mean_true_rank,median_true_rank,mean_cos_true,mean_cos_pred
0,clip_oklch_129c_dim128_t0.03_lr0.0003_wd0.0,oklch,129,128,0.03,0.0003,0.00,0.426357,55,129,2.046512,2.0,0.472566,0.494272
1,clip_oklch_129c_dim128_t0.03_lr0.0003_wd0.01,oklch,129,128,0.03,0.0003,0.01,0.403101,52,129,2.054264,2.0,0.509534,0.534353
2,clip_oklch_129c_dim128_t0.03_lr0.0003_wd0.1,oklch,129,128,0.03,0.0003,0.10,0.379845,49,129,2.255814,2.0,0.585007,0.626229
3,clip_oklch_129c_dim128_t0.03_lr0.001_wd0.0,oklch,129,128,0.03,0.0010,0.00,0.434109,56,129,2.069767,2.0,0.468191,0.489861
4,clip_oklch_129c_dim128_t0.03_lr0.001_wd0.01,oklch,129,128,0.03,0.0010,0.01,0.387597,50,129,2.116279,2.0,0.511239,0.537058


In [16]:
# Merge with original training results for a complete picture
df_orig = results.rename(columns={'Experiment': 'experiment'})
df_merged = df_all.merge(df_orig[['experiment', 'Rank', 'R_at_1', 'R_at_5', 'R_at_10',
                                    'Median_Rank', 'Delta_E', 'Temperature']],
                          on='experiment', how='left', suffixes=('', '_train'))

# Sort by centroid accuracy descending
df_merged = df_merged.sort_values('centroid_accuracy', ascending=False).reset_index(drop=True)
df_merged.index.name = 'centroid_rank'
df_merged.index += 1  # 1-based

# Save to CSV
out_path = PROJECT_ROOT / '5th_experiments' / 'centroid_eval_results.csv'
df_merged.to_csv(out_path)
print(f'Saved {len(df_merged)} rows to {out_path}')
print()

# Summary stats
print('=== Centroid Accuracy Summary ===')
for nc in [15, 129]:
    sub = df_merged[df_merged['num_colors'] == nc]
    print(f'\n{nc}-color ({len(sub)} models):')
    print(f'  Accuracy: mean={sub["centroid_accuracy"].mean():.3f}  '
          f'min={sub["centroid_accuracy"].min():.3f}  '
          f'max={sub["centroid_accuracy"].max():.3f}')
    print(f'  Mean True Rank: mean={sub["mean_true_rank"].mean():.1f}  '
          f'min={sub["mean_true_rank"].min():.1f}  '
          f'max={sub["mean_true_rank"].max():.1f}')
    for cs in ['oklch', 'rgb']:
        s = sub[sub['color_space'] == cs]
        print(f'  [{cs}] acc mean={s["centroid_accuracy"].mean():.3f}  '
              f'max={s["centroid_accuracy"].max():.3f}')

display(df_merged.head(20))

Saved 432 rows to c:\Users\LuizMarques\Downloads\Projects\Projects\colorsurvey\5th_experiments\centroid_eval_results.csv

=== Centroid Accuracy Summary ===

15-color (216 models):
  Accuracy: mean=0.796  min=0.667  max=0.867
  Mean True Rank: mean=1.2  min=1.1  max=1.5
  [oklch] acc mean=0.791  max=0.867
  [rgb] acc mean=0.801  max=0.867

129-color (216 models):
  Accuracy: mean=0.397  min=0.186  max=0.465
  Mean True Rank: mean=2.2  min=2.0  max=4.1
  [oklch] acc mean=0.390  max=0.457
  [rgb] acc mean=0.404  max=0.465


,experiment,color_space,num_colors,embed_dim,temperature,lr,weight_decay,centroid_accuracy,correct,total_classes,...,median_true_rank,mean_cos_true,mean_cos_pred,Rank,R_at_1,R_at_5,R_at_10,Median_Rank,Delta_E,Temperature
centroid_rank,,,,,,,,,,,,,,,,,,,,,
1,clip_rgb_15c_dim16_t0.15_lr0.003_wd0.01,rgb,15,16,0.15,0.0030,0.01,0.866667,13,15,...,1.0,0.629513,0.666147,248,0.0028,0.0120,0.0240,301.0,0.4070,0.0801
2,clip_rgb_15c_dim16_t0.03_lr0.003_wd0.0,rgb,15,16,0.03,0.0030,0.00,0.866667,13,15,...,1.0,0.633800,0.662713,231,0.0028,0.0122,0.0238,303.0,0.4157,0.0638
3,clip_rgb_15c_dim16_t0.07_lr0.001_wd0.1,rgb,15,16,0.07,0.0010,0.10,0.866667,13,15,...,1.0,0.679280,0.722993,355,0.0024,0.0122,0.0236,301.0,0.3902,0.0984
4,clip_rgb_15c_dim16_t0.03_lr0.003_wd0.01,rgb,15,16,0.03,0.0030,0.01,0.866667,13,15,...,1.0,0.623640,0.660687,382,0.0024,0.0120,0.0242,303.0,0.4021,0.0798
5,clip_rgb_15c_dim32_t0.03_lr0.0003_wd0.0,rgb,15,32,0.03,0.0003,0.00,0.866667,13,15,...,1.0,0.497253,0.515580,239,0.0028,0.0118,0.0234,302.0,0.4006,0.0440
6,clip_oklch_15c_dim16_t0.07_lr0.001_wd0.0,oklch,15,16,0.07,0.0010,0.00,0.866667,13,15,...,1.0,0.715593,0.745067,427,0.0020,0.0118,0.0232,304.0,0.2641,0.0647
7,clip_oklch_15c_dim128_t0.15_lr0.003_wd0.0,oklch,15,128,0.15,0.0030,0.00,0.866667,13,15,...,1.0,0.682393,0.711907,423,0.0022,0.0118,0.0228,304.0,0.2688,0.0633
8,clip_oklch_15c_dim16_t0.07_lr0.0003_wd0.0,oklch,15,16,0.07,0.0003,0.00,0.866667,13,15,...,1.0,0.702700,0.730827,334,0.0026,0.0118,0.0226,302.0,0.2623,0.0649
9,clip_oklch_15c_dim128_t0.07_lr0.003_wd0.01,oklch,15,128,0.07,0.0030,0.01,0.866667,13,15,...,1.0,0.691047,0.726400,407,0.0022,0.0116,0.0226,303.0,0.2790,0.0767


## 12. Best Models Side-by-Side

In [17]:
# Show the best 15-color and best 129-color models together
print(f'=== Best 15-Color Model: {exp_15} ===')
print(f'  R@1={best_15["R_at_1"]:.4f}  Delta_E={best_15["Delta_E"]:.4f}  Color_Space={cs_15}  Embed_Dim={dim_15}')
print()
print(f'=== Best 129-Color Model: {exp_129} ===')
print(f'  R@1={best_129["R_at_1"]:.4f}  Delta_E={best_129["Delta_E"]:.4f}  Color_Space={cs_129}  Embed_Dim={dim_129}')
print()

display_results(df_15.sort_values('True Name'), f'Best 15-Color Model: {exp_15}')
display_results(df_129.sort_values('True Name'), f'Best 129-Color Model: {exp_129}')

=== Best 15-Color Model: clip_oklch_15c_dim64_t0.07_lr0.001_wd0.1 ===
  R@1=0.0030  Delta_E=0.2831  Color_Space=oklch  Embed_Dim=64

=== Best 129-Color Model: clip_rgb_129c_dim32_t0.03_lr0.003_wd0.0 ===
  R@1=0.0074  Delta_E=0.3089  Color_Space=rgb  Embed_Dim=32



Input RGB,True Name,Output Color Name,Output RGB (REFERENCE),Cos Sim (pred),Cos Sim (true),True Rank
"(55, 97, 198)",blue,blue,"(55, 97, 198)",0.6880,0.6880,1
"(136, 90, 44)",brown,brown,"(136, 90, 44)",0.9368,0.9368,1
"(74, 185, 68)",green,green,"(74, 185, 68)",0.7684,0.7684,1
"(144, 148, 143)",grey,grey,"(144, 148, 143)",0.9255,0.9255,1
"(99, 184, 223)",light blue,sky blue,"(90, 181, 226)",0.4843,0.4463,3
"(118, 224, 107)",light green,green,"(74, 185, 68)",0.5862,0.3646,2
"(200, 39, 152)",magenta,magenta,"(200, 39, 152)",0.5031,0.5031,1
"(228, 122, 38)",orange,orange,"(228, 122, 38)",0.8783,0.8783,1
"(227, 91, 169)",pink,pink,"(227, 91, 169)",0.7272,0.7272,1
"(142, 53, 173)",purple,purple,"(142, 53, 173)",0.7087,0.7087,1


Input RGB,True Name,Output Color Name,Output RGB (REFERENCE),Cos Sim (pred),Cos Sim (true),True Rank
"(62, 206, 189)",aqua,turquoise,"(54, 195, 178)",0.3528,0.3457,5
"(58, 199, 177)",aquamarine,teal,"(57, 179, 159)",0.3553,0.3367,6
"(97, 123, 49)",army green,army green,"(97, 123, 49)",0.3913,0.3913,1
"(121, 193, 228)",baby blue,baby blue,"(121, 193, 228)",0.3729,0.3729,1
"(211, 192, 133)",beige,tan,"(203, 172, 103)",0.4094,0.4045,2
"(36, 33, 35)",black,black,"(36, 33, 35)",0.4748,0.4748,1
"(55, 97, 198)",blue,blue,"(55, 97, 198)",0.4315,0.4315,1
"(108, 138, 159)",blue gray,blue grey,"(110, 140, 161)",0.4141,0.4121,3
"(51, 169, 140)",blue green,teal,"(57, 179, 159)",0.3686,0.3432,2
"(110, 140, 161)",blue grey,blue grey,"(110, 140, 161)",0.4133,0.4133,1
